## dataset info

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("shortened_result-val.csv")
df.head()


,subject_id,visit,entry_age,CDGLOBAL,MMSCORE,DIAGNOSIS,TOTSCORE,num_visits,num_dirty_rows,num_dirty_mmse,num_dirty_cdglobal,num_dirty_totscore
0,941_S_10002,bl,72.95,NaN,NaN,2.0,9.00,4,2,1,1,1
1,941_S_10002,m12,72.95,0.5,24.0,2.0,2.33,4,2,1,1,1
2,941_S_10002,m24,72.95,0.5,24.0,2.0,5.00,4,2,1,1,1
3,941_S_10002,sc,72.95,0.5,27.0,2.0,NaN,4,2,1,1,1
4,941_S_7074,bl,70.90,NaN,NaN,1.0,3.00,5,3,2,2,2


In [4]:
print(df.shape)
df.columns.tolist()

(22168, 12)


['subject_id',
 'visit',
 'entry_age',
 'CDGLOBAL',
 'MMSCORE',
 'DIAGNOSIS',
 'TOTSCORE',
 'num_visits',
 'num_dirty_rows',
 'num_dirty_mmse',
 'num_dirty_cdglobal',
 'num_dirty_totscore']

In [5]:
df.dtypes

subject_id             object
visit                  object
entry_age             float64
CDGLOBAL              float64
MMSCORE               float64
DIAGNOSIS             float64
TOTSCORE              float64
num_visits              int64
num_dirty_rows          int64
num_dirty_mmse          int64
num_dirty_cdglobal      int64
num_dirty_totscore      int64
dtype: object

In [6]:
df["subject_id"].isna().sum()

0

In [7]:
df["subject_id"].nunique()

3034

In [8]:
df["visit"].value_counts(dropna=False)

visit
sc       3034
bl       2944
m12      2091
m24      1637
m06      1573
m18      1312
m36      1154
scmri     976
m48       888
m30       845
m03       783
m60       513
m72       502
m84       435
m78       419
m42       400
m66       373
m90       298
m54       297
m96       287
m108      231
m120      155
m102      110
m132      103
m144       89
m156       80
nv         78
m114       73
m126       71
m138       59
AUT        58
m150       57
m162       51
m168       50
m180       28
m174       22
m186       16
m192       12
NaN        12
m222       10
m228        8
m204        7
m198        7
m216        7
m234        6
m210        4
m240        2
uns1        1
Name: count, dtype: int64

In [9]:
target_cols = ["CDGLOBAL", "MMSCORE", "TOTSCORE"]

df[target_cols].isna().sum()

CDGLOBAL    9299
MMSCORE     9597
TOTSCORE    9747
dtype: int64

In [10]:
df[target_cols].isna().mean() * 100


CDGLOBAL    41.947853
MMSCORE     43.292133
TOTSCORE    43.968784
dtype: float64

In [11]:
df[target_cols].describe()


,CDGLOBAL,MMSCORE,TOTSCORE
count,12869.000000,12571.000000,12421.000000
mean,0.425635,26.921327,10.545333
std,0.472210,3.750929,8.083320
min,0.000000,0.000000,0.000000
25%,0.000000,26.000000,5.000000
50%,0.500000,28.000000,8.330000
75%,0.500000,30.000000,13.670000
max,3.000000,30.000000,70.000000


In [12]:
df["DIAGNOSIS"].value_counts(dropna=False)


DIAGNOSIS
2.0    8897
1.0    8800
3.0    4471
Name: count, dtype: int64

In [13]:
df.groupby("visit")[target_cols].apply(lambda x: x.isna().mean())


,CDGLOBAL,MMSCORE,TOTSCORE
visit,,,
AUT,1.000000,1.000000,1.000000
bl,1.000000,1.000000,0.005774
m03,1.000000,1.000000,1.000000
m06,0.015893,0.010172,0.012079
m102,0.372727,0.472727,0.481818
m108,0.272727,0.393939,0.406926
m114,0.356164,0.424658,0.424658
m12,0.166906,0.170253,0.172645
m120,0.167742,0.270968,0.277419


## CLEAN VISIT NORMALIZATION

In [14]:
MAX_MONTH = 120

INVALID_VISITS = {"scmri", "AUT", "nv", "uns1"}


In [15]:
def visit_to_month(visit):
    if pd.isna(visit):
        return np.nan

    visit = str(visit).strip().lower()

    if visit in INVALID_VISITS:
        return np.nan

    if visit == "bl":
        return 0
    if visit == "sc":
        return 1
    if visit.startswith("m"):
        try:
            return int(visit[1:])
        except ValueError:
            return np.nan

    return np.nan


In [16]:
df["visit_month"] = df["visit"].apply(visit_to_month)

# Drop invalid or extreme visits
df_clean = df[
    (df["visit_month"].notna()) &
    (df["visit_month"] <= MAX_MONTH)
].copy()

df_clean.shape


(20354, 13)

In [17]:
df_clean = (
    df_clean
    .sort_values(by=["subject_id", "visit_month"])
    .reset_index(drop=True)
)

df_clean.head(10)


,subject_id,visit,entry_age,CDGLOBAL,MMSCORE,DIAGNOSIS,TOTSCORE,num_visits,num_dirty_rows,num_dirty_mmse,num_dirty_cdglobal,num_dirty_totscore,visit_month
0,002_S_0295,bl,84.84,NaN,NaN,1.0,3.00,14,7,6,6,6,0.0
1,002_S_0295,sc,84.84,0.0,28.0,1.0,NaN,14,7,6,6,6,1.0
2,002_S_0295,m06,84.84,0.0,28.0,1.0,5.33,14,7,6,6,6,6.0
3,002_S_0295,m12,84.84,0.0,30.0,1.0,4.67,14,7,6,6,6,12.0
4,002_S_0295,m18,84.84,NaN,NaN,1.0,NaN,14,7,6,6,6,18.0
5,002_S_0295,m24,84.84,0.0,29.0,1.0,3.67,14,7,6,6,6,24.0
6,002_S_0295,m30,84.84,NaN,NaN,1.0,NaN,14,7,6,6,6,30.0
7,002_S_0295,m36,84.84,0.0,28.0,1.0,3.67,14,7,6,6,6,36.0
8,002_S_0295,m42,84.84,NaN,NaN,1.0,NaN,14,7,6,6,6,42.0
9,002_S_0295,m48,84.84,0.0,26.0,1.0,5.00,14,7,6,6,6,48.0


In [18]:
df_clean["visit"].value_counts().head(15)


visit
sc     3034
bl     2944
m12    2091
m24    1637
m06    1573
m18    1312
m36    1154
m48     888
m30     845
m03     783
m60     513
m72     502
m84     435
m78     419
m42     400
Name: count, dtype: int64

In [19]:
df_clean[["CDGLOBAL", "MMSCORE", "TOTSCORE"]].isna().mean() * 100


CDGLOBAL    39.147096
MMSCORE     40.291835
TOTSCORE    40.984573
dtype: float64

## TRAJECTORY PROFILING

In [20]:
TARGET_COLS = ["CDGLOBAL", "MMSCORE", "TOTSCORE"]


In [21]:
def count_valid(series):
    return series.notna().sum()

def is_constant(series):
    vals = series.dropna().unique()
    return len(vals) == 1

def is_monotonic(series):
    vals = series.dropna().values
    if len(vals) < 3:
        return False
    return np.all(np.diff(vals) >= 0) or np.all(np.diff(vals) <= 0)



In [22]:
profiles = []

for pid, g in df_clean.groupby("subject_id"):
    profile = {"subject_id": pid}
    
    for col in TARGET_COLS:
        s = g[col]
        profile[f"{col}_valid"] = count_valid(s)
        profile[f"{col}_constant"] = is_constant(s)
        profile[f"{col}_monotonic"] = is_monotonic(s)
    
    profile["num_visits"] = len(g)
    profiles.append(profile)

traj_profile = pd.DataFrame(profiles)
traj_profile.head()


,subject_id,CDGLOBAL_valid,CDGLOBAL_constant,CDGLOBAL_monotonic,MMSCORE_valid,MMSCORE_constant,MMSCORE_monotonic,TOTSCORE_valid,TOTSCORE_constant,TOTSCORE_monotonic,num_visits
0,002_S_0295,8,True,True,8,False,False,8,False,False,14
1,002_S_0413,11,True,True,11,False,False,11,False,False,19
2,002_S_0559,5,True,True,5,False,False,5,False,False,8
3,002_S_0619,4,False,True,4,False,False,4,False,False,6
4,002_S_0685,9,True,True,8,False,False,8,False,False,18


In [23]:
traj_profile[[c for c in traj_profile.columns if "valid" in c]].describe()


,CDGLOBAL_valid,MMSCORE_valid,TOTSCORE_valid
count,3034.000000,3034.000000,3034.00000
mean,4.082399,4.005603,3.95913
std,2.961641,2.904509,2.93848
min,0.000000,0.000000,0.00000
25%,1.000000,1.000000,1.00000
50%,4.000000,3.000000,3.00000
75%,6.000000,6.000000,6.00000
max,13.000000,13.000000,13.00000


In [24]:
traj_profile[[c for c in traj_profile.columns if "constant" in c]].mean() * 100


CDGLOBAL_constant    66.710613
MMSCORE_constant     32.201714
TOTSCORE_constant    25.082399
dtype: float64

In [25]:
traj_profile[[c for c in traj_profile.columns if "monotonic" in c]].mean() * 100


CDGLOBAL_monotonic    52.900461
MMSCORE_monotonic     16.117337
TOTSCORE_monotonic     9.492419
dtype: float64

In [26]:
def trajectory_type(row, col):
    if row[f"{col}_valid"] >= 3 and row[f"{col}_monotonic"]:
        return "monotonic"
    if row[f"{col}_valid"] >= 2 and row[f"{col}_constant"]:
        return "constant"
    if row[f"{col}_valid"] >= 2:
        return "sparse_trend"
    return "too_sparse"


In [27]:
for col in TARGET_COLS:
    traj_profile[f"{col}_traj_type"] = traj_profile.apply(
        lambda r: trajectory_type(r, col), axis=1
    )

traj_profile.head()


,subject_id,CDGLOBAL_valid,CDGLOBAL_constant,CDGLOBAL_monotonic,MMSCORE_valid,MMSCORE_constant,MMSCORE_monotonic,TOTSCORE_valid,TOTSCORE_constant,TOTSCORE_monotonic,num_visits,CDGLOBAL_traj_type,MMSCORE_traj_type,TOTSCORE_traj_type
0,002_S_0295,8,True,True,8,False,False,8,False,False,14,monotonic,sparse_trend,sparse_trend
1,002_S_0413,11,True,True,11,False,False,11,False,False,19,monotonic,sparse_trend,sparse_trend
2,002_S_0559,5,True,True,5,False,False,5,False,False,8,monotonic,sparse_trend,sparse_trend
3,002_S_0619,4,False,True,4,False,False,4,False,False,6,monotonic,sparse_trend,sparse_trend
4,002_S_0685,9,True,True,8,False,False,8,False,False,18,monotonic,sparse_trend,sparse_trend


In [28]:
for col in TARGET_COLS:
    print(f"\n{col} trajectory distribution:")
    print(traj_profile[f"{col}_traj_type"].value_counts(normalize=True) * 100)



CDGLOBAL trajectory distribution:
CDGLOBAL_traj_type
monotonic       52.900461
too_sparse      27.059987
sparse_trend    12.392881
constant         7.646671
Name: proportion, dtype: float64

MMSCORE trajectory distribution:
MMSCORE_traj_type
sparse_trend    53.790376
too_sparse      27.653263
monotonic       16.117337
constant         2.439024
Name: proportion, dtype: float64

TOTSCORE trajectory distribution:
TOTSCORE_traj_type
sparse_trend    62.162162
too_sparse      27.851022
monotonic        9.492419
constant         0.494397
Name: proportion, dtype: float64


## BUILD 4-VISIT LONGITUDINAL SAMPLES

In [29]:
def select_earliest_4(g):
    return g.sort_values("visit_month").head(4)

df_4 = (
    df_clean
    .groupby("subject_id", group_keys=False)
    .apply(select_earliest_4)
    .reset_index(drop=True)
)

df_4.head(10)


C:\Users\Mihit Singasane\AppData\Local\Temp\ipykernel_28288\3495377696.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_earliest_4)


,subject_id,visit,entry_age,CDGLOBAL,MMSCORE,DIAGNOSIS,TOTSCORE,num_visits,num_dirty_rows,num_dirty_mmse,num_dirty_cdglobal,num_dirty_totscore,visit_month
0,002_S_0295,bl,84.84,NaN,NaN,1.0,3.00,14,7,6,6,6,0.0
1,002_S_0295,sc,84.84,0.0,28.0,1.0,NaN,14,7,6,6,6,1.0
2,002_S_0295,m06,84.84,0.0,28.0,1.0,5.33,14,7,6,6,6,6.0
3,002_S_0295,m12,84.84,0.0,30.0,1.0,4.67,14,7,6,6,6,12.0
4,002_S_0413,bl,76.34,NaN,NaN,1.0,3.33,24,11,10,10,10,0.0
5,002_S_0413,sc,76.34,0.0,29.0,1.0,NaN,24,11,10,10,10,1.0
6,002_S_0413,m06,76.34,0.0,29.0,1.0,7.33,24,11,10,10,10,6.0
7,002_S_0413,m12,76.34,0.0,29.0,1.0,2.33,24,11,10,10,10,12.0
8,002_S_0559,bl,79.36,NaN,NaN,1.0,6.00,8,4,3,3,3,0.0
9,002_S_0559,sc,79.36,0.0,30.0,1.0,NaN,8,4,3,3,3,1.0


In [30]:
df_4.groupby("subject_id").size().value_counts()


4    2056
2     579
3     309
1      90
Name: count, dtype: int64

In [31]:
df_4 = df_4.merge(
    traj_profile[
        ["subject_id"] +
        [f"{c}_traj_type" for c in TARGET_COLS]
    ],
    on="subject_id",
    how="left"
)

df_4.head()


,subject_id,visit,entry_age,CDGLOBAL,MMSCORE,DIAGNOSIS,TOTSCORE,num_visits,num_dirty_rows,num_dirty_mmse,num_dirty_cdglobal,num_dirty_totscore,visit_month,CDGLOBAL_traj_type,MMSCORE_traj_type,TOTSCORE_traj_type
0,002_S_0295,bl,84.84,NaN,NaN,1.0,3.00,14,7,6,6,6,0.0,monotonic,sparse_trend,sparse_trend
1,002_S_0295,sc,84.84,0.0,28.0,1.0,NaN,14,7,6,6,6,1.0,monotonic,sparse_trend,sparse_trend
2,002_S_0295,m06,84.84,0.0,28.0,1.0,5.33,14,7,6,6,6,6.0,monotonic,sparse_trend,sparse_trend
3,002_S_0295,m12,84.84,0.0,30.0,1.0,4.67,14,7,6,6,6,12.0,monotonic,sparse_trend,sparse_trend
4,002_S_0413,bl,76.34,NaN,NaN,1.0,3.33,24,11,10,10,10,0.0,monotonic,sparse_trend,sparse_trend


In [32]:
for col in TARGET_COLS:
    df_4[f"{col}_is_missing"] = df_4[col].isna()


In [33]:
df_4[[f"{c}_is_missing" for c in TARGET_COLS]].mean() * 100


CDGLOBAL_is_missing    40.167324
MMSCORE_is_missing     40.782768
TOTSCORE_is_missing    41.907876
dtype: float64

In [ ]:
def valid_count_local(series):
    return series.notna().sum()

def resolve_strategy(global_traj_type, local_valid_count):
    """
    Downgrade strategy if not enough local signal.
    """
    if global_traj_type == "constant" and local_valid_count >= 1:
        return "constant"

    if global_traj_type == "monotonic" and local_valid_count >= 2:
        return "monotonic"

    if local_valid_count >= 1:
        return "sparse_trend"

    return "too_sparse"

def impute_column_patientwise(g, col, strategy):
    months = g["visit_month"]
    values = g[col].copy()

    if strategy == "constant":
        const_val = values.dropna().iloc[0]
        return values.fillna(const_val)

    if strategy == "monotonic":
        interp = pd.Series(values.values, index=months)\
                   .interpolate(method="index", limit_area="inside")
        return interp.ffill().bfill().values

    if strategy == "sparse_trend":
        return values.ffill().bfill()

    return values  # too_sparse → leave NaN


In [35]:
df_imputed = []

for pid, g in df_4.groupby("subject_id"):
    g = g.sort_values("visit_month").copy()

    for col in TARGET_COLS:
        traj_type = g[f"{col}_traj_type"].iloc[0]
        g[col] = impute_column_patientwise(g, col, traj_type)

    df_imputed.append(g)

df_imputed = pd.concat(df_imputed).reset_index(drop=True)
df_imputed.head(10)


,subject_id,visit,entry_age,CDGLOBAL,MMSCORE,DIAGNOSIS,TOTSCORE,num_visits,num_dirty_rows,num_dirty_mmse,num_dirty_cdglobal,num_dirty_totscore,visit_month,CDGLOBAL_traj_type,MMSCORE_traj_type,TOTSCORE_traj_type,CDGLOBAL_is_missing,MMSCORE_is_missing,TOTSCORE_is_missing
0,002_S_0295,bl,84.84,0.0,28.0,1.0,3.00,14,7,6,6,6,0.0,monotonic,sparse_trend,sparse_trend,True,True,False
1,002_S_0295,sc,84.84,0.0,28.0,1.0,3.00,14,7,6,6,6,1.0,monotonic,sparse_trend,sparse_trend,False,False,True
2,002_S_0295,m06,84.84,NaN,28.0,1.0,5.33,14,7,6,6,6,6.0,monotonic,sparse_trend,sparse_trend,False,False,False
3,002_S_0295,m12,84.84,NaN,30.0,1.0,4.67,14,7,6,6,6,12.0,monotonic,sparse_trend,sparse_trend,False,False,False
4,002_S_0413,bl,76.34,NaN,29.0,1.0,3.33,24,11,10,10,10,0.0,monotonic,sparse_trend,sparse_trend,True,True,False
5,002_S_0413,sc,76.34,NaN,29.0,1.0,3.33,24,11,10,10,10,1.0,monotonic,sparse_trend,sparse_trend,False,False,True
6,002_S_0413,m06,76.34,0.0,29.0,1.0,7.33,24,11,10,10,10,6.0,monotonic,sparse_trend,sparse_trend,False,False,False
7,002_S_0413,m12,76.34,NaN,29.0,1.0,2.33,24,11,10,10,10,12.0,monotonic,sparse_trend,sparse_trend,False,False,False
8,002_S_0559,bl,79.36,NaN,30.0,1.0,6.00,8,4,3,3,3,0.0,monotonic,sparse_trend,sparse_trend,True,True,False
9,002_S_0559,sc,79.36,NaN,30.0,1.0,6.00,8,4,3,3,3,1.0,monotonic,sparse_trend,sparse_trend,False,False,True


In [36]:
df_imputed[TARGET_COLS].isna().mean() * 100


CDGLOBAL    61.717473
MMSCORE     18.886431
TOTSCORE    12.145399
dtype: float64

In [37]:
def imputation_confidence(row):
    if (
        row["CDGLOBAL_traj_type"] == "too_sparse" or
        row["MMSCORE_traj_type"] == "too_sparse" or
        row["TOTSCORE_traj_type"] == "too_sparse"
    ):
        return "low"
    return "high"

df_imputed["imputation_confidence"] = df_imputed.apply(imputation_confidence, axis=1)
df_imputed["imputation_confidence"].value_counts(normalize=True) * 100


imputation_confidence
high    82.209828
low     17.790172
Name: proportion, dtype: float64